<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Bank ClickStream - GNN for Feature Engineering: Multiclass Apply Event Prediction
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style="font-size:20px;font-family:Arial"><b>Introduction</b></p>
<p style="font-size:16px;font-family:Arial"> In this notebook we willPredicting which specific banking product a user will apply for using Graph Neural Networks (GraphSAGE) as a feature
engineering layer feeding into XGBoost.
 <ol style="font-size:16px;font-family:Arial"> 8 Target Class  
     <li>No Application (Other)</li>
<li>ApplyCreditCard</li>
<li>ApplySavingsAccount</li>
<li>ApplyAutoLoan</li>
<li>ApplyCheckingAccount</li>
     <li>ApplyPersonalLoan</li>
     <li>ApplyMortgage</li>
     <li>ApplyTeenChecking</li>
     </ol>
     
<p style="font-size:18px;font-family:Arial">The overall processing pipeline follows the sequence as below:</p> 
<img src="./images/gnn_pipeline.png" alt="gnn" style="width:100%; border: 4px solid #404040; border-radius: 10px;" />
<br>

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>1. Setup and Imports </b></p>

In [ ]:
!pip install torch_geometric

<div class="alert alert-block alert-info">
<p style = 'font-size:16px;font-family:Arial'><b>Please</b><i> restart the kernel after executing the above cell to include/update these libraries into memory for this kernel. The simplest way to restart the Kernel is by typing zero zero: <b> 0 0</b></i> and then clicking <b>Restart</b>.</p>
</div>

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool, global_max_pool

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle

from teradataml import *

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("✅ Using Apple M2 GPU (MPS)")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"✅ Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ Using CPU")


<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>2. Robust Label Encoder</b></p>


In [ ]:
class RobustLabelEncoder:
    """Handles unseen labels by mapping to <UNK> (index 0)."""
    
    def __init__(self):
        self.classes_ = None
        self.class_to_idx = None
        
    def fit(self, values):
        unique = sorted(set(values))
        self.classes_ = ['<UNK>'] + list(unique)
        self.class_to_idx = {c: i for i, c in enumerate(self.classes_)}
        return self
    
    def transform(self, values):
        return np.array([self.class_to_idx.get(v, 0) for v in values])
    
    @property
    def vocab_size(self):
        return len(self.classes_)

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>3. Connect to TeradataCloud</b></p>
<p style = 'font-size:16px;font-family:Arial'>Connect to TeradataCloud using <code>create_context</code> from the teradataml Python library. </p>

In [ ]:
print("Checking if this environment is ready to connect to TeradataCloud Lake...")

if os.path.exists("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env"):
    print("Your environment parameter file exist.  Please proceed with this use case.")
    # Load all the variables from the .env file into a dictionary
    env_vars = dotenv_values("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env")
    # Create the Context
    eng = create_context(host=env_vars.get("host"), username=env_vars.get("username"), password=env_vars.get("my_variable"))
    execute_sql('''SET query_band='DEMO=6._Bank_Click_Stream_-_Outcome_Prediction_Model_with_GNN_Feature_Engg.ipynb;' UPDATE FOR SESSION;''')
    print("Connected to TeradataCloud with:", eng)
else:
    print("Your environment has not been prepared for connecting to TeradataCloud.")
    print("Please contact the support team.")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>4. Load Data & Create Multiclass Target</b></p>

In [ ]:
tdf = DataFrame(in_schema('DEMO_Bank','Session_Events'))

In [ ]:
df = tdf[['UserID', 'SessionID', 'Event']].to_pandas(all_rows = True)

print(f"Shape: {df.shape}")
print(f"Events: {df['Event'].nunique()}")
print(f"Sessions: {df.groupby(['UserID', 'SessionID']).ngroups}")

In [ ]:
# Find all Apply* events
apply_events = sorted(df[df['Event'].str.startswith('Apply')]['Event'].unique())
print(f"\nApply events found ({len(apply_events)}):")
for i, e in enumerate(apply_events):
    count = (df['Event'] == e).sum()
    print(f"  {i+1}. {e}: {count}")

In [ ]:
# Create target class mapping
# 0 = No Application, 1-N = Specific Apply events

TARGET_CLASSES = ['NoApplication'] + apply_events
TARGET_TO_IDX = {t: i for i, t in enumerate(TARGET_CLASSES)}
IDX_TO_TARGET = {i: t for t, i in TARGET_TO_IDX.items()}

NUM_CLASSES = len(TARGET_CLASSES)

print(f"\nTarget Classes ({NUM_CLASSES}):")
for idx, name in IDX_TO_TARGET.items():
    print(f"  {idx}: {name}")

In [ ]:
def get_session_target(session_events):
    """
    Determine the target class for a session.
    
    Logic:
    - If session has Apply event(s), return the FIRST Apply event encountered
    - If no Apply event, return 0 (NoApplication)
    
    Note: If you want to handle multiple Apply events differently,
    you could return the LAST one, or create multi-label targets.
    """
    for event in session_events:
        if event in TARGET_TO_IDX:
            return TARGET_TO_IDX[event]
    return 0  # NoApplication


# Create session-level targets
session_targets = []
for (uid, sid), group in df.groupby(['UserID', 'SessionID'], sort=False):
    events = group['Event'].tolist()
    target = get_session_target(events)
    session_targets.append({'UserID': uid, 'SessionID': sid, 'target': target})

session_labels = pd.DataFrame(session_targets)

print("\nTarget Distribution:")
target_counts = session_labels['target'].value_counts().sort_index()
for idx, count in target_counts.items():
    pct = count / len(session_labels) * 100
    print(f"  {idx} ({IDX_TO_TARGET[idx]}): {count} ({pct:.2f}%)")

In [ ]:
# Create event encoder (exclude Apply events from input features)
# We don't want the model to "cheat" by seeing Apply events in the sequence

# Option 1: Keep Apply events (model sees full sequence including outcome)
# Option 2: Remove Apply events (model predicts based on journey BEFORE application)

# Let's use Option 2 - more realistic for prediction
EXCLUDE_APPLY_FROM_INPUT = True

if EXCLUDE_APPLY_FROM_INPUT:
    input_events = df[~df['Event'].str.startswith('Apply')]['Event'].unique()
    print(f"Input events (excluding Apply*): {len(input_events)}")
else:
    input_events = df['Event'].unique()
    print(f"Input events (all): {len(input_events)}")

event_encoder = RobustLabelEncoder()
event_encoder.fit(input_events)

EVENT_VOCAB = event_encoder.vocab_size
print(f"Event vocabulary: {EVENT_VOCAB}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>5. Graph Construction</b></p>

In [ ]:
class SimpleGraphBuilder:
    """
    Builds graphs using only events (excluding Apply events from features).
    """
    
    def __init__(self, event_encoder, exclude_apply=True):
        self.event_encoder = event_encoder
        self.exclude_apply = exclude_apply
        
    def build_graph(self, session_df, target=0):
        session_df = session_df.reset_index(drop=True)
        
        # Filter out Apply events if needed
        if self.exclude_apply:
            session_df = session_df[~session_df['Event'].str.startswith('Apply')].reset_index(drop=True)
        
        n = len(session_df)
        
        if n == 0:
            # Session only had Apply event(s), no journey
            # Create minimal graph with single <UNK> node
            return Data(
                x=torch.tensor([[0, 0.0]], dtype=torch.float),
                edge_index=torch.tensor([[0], [0]], dtype=torch.long),
                y=torch.tensor([target], dtype=torch.long),
                num_nodes=1,
                event_ids=torch.tensor([0], dtype=torch.long)
            )
        
        event_ids = self.event_encoder.transform(session_df['Event'].values)
        positions = np.arange(n) / max(n - 1, 1)
        node_features = np.column_stack([event_ids, positions])
        
        if n > 1:
            src = list(range(n-1)) + list(range(1, n))
            dst = list(range(1, n)) + list(range(n-1))
            edge_index = torch.tensor([src, dst], dtype=torch.long)
        else:
            edge_index = torch.tensor([[0], [0]], dtype=torch.long)
        
        data = Data(
            x=torch.tensor(node_features, dtype=torch.float),
            edge_index=edge_index,
            y=torch.tensor([target], dtype=torch.long),
            num_nodes=n
        )
        data.event_ids = torch.tensor(event_ids, dtype=torch.long)
        
        return data


graph_builder = SimpleGraphBuilder(event_encoder, exclude_apply=EXCLUDE_APPLY_FROM_INPUT)
print("✅ Graph builder ready")

In [ ]:
# Build all graphs
def build_all_graphs(df, session_labels, graph_builder):
    graphs, labels, session_ids = [], [], []
    grouped = df.groupby(['UserID', 'SessionID'], sort=False)
    target_lookup = session_labels.set_index(['UserID', 'SessionID'])['target'].to_dict()
    
    for (uid, sid), sdf in tqdm(grouped, desc="Building graphs"):
        target = target_lookup.get((uid, sid), 0)
        graph = graph_builder.build_graph(sdf, target)
        if graph is not None:
            graphs.append(graph)
            labels.append(target)
            session_ids.append((uid, sid))
    
    return graphs, labels, session_ids


graphs, labels, session_ids = build_all_graphs(df, session_labels, graph_builder)

print(f"\n✅ Built {len(graphs)} graphs")
print(f"\nClass distribution:")
for idx in range(NUM_CLASSES):
    count = labels.count(idx)
    print(f"  {idx} ({IDX_TO_TARGET[idx]}): {count}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>6. GNN Encoder</b></p>

In [ ]:
class GNNEncoder(nn.Module):
    """GraphSAGE encoder for session graphs."""
    
    def __init__(self, event_vocab, embed_dim=64, hidden_dim=64, num_layers=3, output_dim=64, dropout=0.2):
        super().__init__()
        
        self.event_emb = nn.Embedding(event_vocab, embed_dim, padding_idx=0)
        input_dim = embed_dim + 1  # embedding + position
        
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        self.convs.append(SAGEConv(input_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        
        self.dropout = nn.Dropout(dropout)
        
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
        
        self.output_dim = output_dim
        
    def forward(self, data):
        e = self.event_emb(data.event_ids)
        pos = data.x[:, 1:2]
        x = torch.cat([e, pos], dim=1)
        
        edge_index, batch = data.edge_index, data.batch
        
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1)
        
        return self.readout(x)


class GNNClassifier(nn.Module):
    """Encoder + multiclass classification head."""
    
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(encoder.output_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, data):
        embedding = self.encoder(data)
        logits = self.classifier(embedding)
        return logits, embedding


# Initialize
encoder = GNNEncoder(
    event_vocab=EVENT_VOCAB,
    embed_dim=64,
    hidden_dim=64,
    num_layers=3,
    output_dim=64
).to(DEVICE)

model = GNNClassifier(encoder, num_classes=NUM_CLASSES).to(DEVICE)

print(f"✅ Model initialized")
print(f"   Classes: {NUM_CLASSES}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>7. Training</b></p>

In [ ]:
# Split data (stratified by target class)
train_idx, test_idx = train_test_split(
    range(len(graphs)), test_size=0.2, stratify=labels, random_state=SEED
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.15, stratify=[labels[i] for i in train_idx], random_state=SEED
)

train_graphs = [graphs[i] for i in train_idx]
val_graphs = [graphs[i] for i in val_idx]
test_graphs = [graphs[i] for i in test_idx]

BATCH_SIZE = 64
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE)

print(f"Train: {len(train_graphs)} | Val: {len(val_graphs)} | Test: {len(test_graphs)}")

In [ ]:
# Class weights for imbalanced data
train_labels = [g.y.item() for g in train_graphs]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES  # Normalize
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print("Class weights:")
for i, w in enumerate(class_weights.cpu().numpy()):
    print(f"  {i} ({IDX_TO_TARGET[i]}): {w:.4f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits, _ = model(batch)
        loss = criterion(logits, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        correct += (logits.argmax(1) == batch.y).sum().item()
        total += batch.num_graphs
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    
    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch)
        loss = criterion(logits, batch.y)
        probs = F.softmax(logits, dim=1)
        
        total_loss += loss.item() * batch.num_graphs
        correct += (logits.argmax(1) == batch.y).sum().item()
        total += batch.num_graphs
        
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    
    # Macro-averaged metrics
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    return {
        'loss': total_loss / total,
        'accuracy': correct / total,
        'preds': all_preds,
        'labels': all_labels,
        'probs': all_probs
    }

In [ ]:
# Training loop
NUM_EPOCHS, PATIENCE = 5, 1
best_val_acc, patience_counter = 0, 0
history = {'train_loss': [], 'val_acc': []}

print("="*60 + "\nTRAINING (Multiclass)\n" + "="*60)

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val = evaluate(model, val_loader, criterion, DEVICE, NUM_CLASSES)
    
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val['accuracy'])
    
    scheduler.step(val['accuracy'])
    
    if val['accuracy'] > best_val_acc:
        best_val_acc = val['accuracy']
        patience_counter = 0
        torch.save(model.state_dict(), 'best_multiclass_gnn.pt')
    else:
        patience_counter += 1
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {train_loss:.4f} | Val Acc: {val['accuracy']:.4f}")
    
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\n✅ Best validation accuracy: {best_val_acc:.4f}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>8. Extract Embeddings & Train XGBoost</b></p>

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_multiclass_gnn.pt'))
model.eval()

@torch.no_grad()
def extract_embeddings(model, graphs, device):
    loader = DataLoader(graphs, batch_size=64, shuffle=False)
    embeddings, labels = [], []
    for batch in loader:
        batch = batch.to(device)
        _, emb = model(batch)
        embeddings.append(emb.cpu().numpy())
        labels.append(batch.y.cpu().numpy())
    return np.vstack(embeddings), np.concatenate(labels)


train_emb, train_lab = extract_embeddings(model, train_graphs, DEVICE)
val_emb, val_lab = extract_embeddings(model, val_graphs, DEVICE)
test_emb, test_lab = extract_embeddings(model, test_graphs, DEVICE)

print(f"✅ Embeddings extracted")
print(f"   Train: {train_emb.shape}")
print(f"   Test: {test_emb.shape}")

In [ ]:
# Train XGBoost multiclass classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=NUM_CLASSES,
    random_state=SEED,
    eval_metric='mlogloss',
    early_stopping_rounds=20,
    n_jobs=-1
)

xgb_model.fit(
    train_emb, train_lab,
    eval_set=[(val_emb, val_lab)],
    verbose=False
)

print("✅ XGBoost trained")

In [ ]:
# Evaluate
test_probs = xgb_model.predict_proba(test_emb)
test_preds = xgb_model.predict(test_emb)

print("="*60)
print("TEST RESULTS (GNN + XGBoost Multiclass)")
print("="*60)

print(f"\nAccuracy: {(test_preds == test_lab).mean():.4f}")

print("\nClassification Report:")
print(classification_report(
    test_lab, test_preds,
    target_names=[IDX_TO_TARGET[i] for i in range(NUM_CLASSES)],
    zero_division=0
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_lab, test_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[IDX_TO_TARGET[i] for i in range(NUM_CLASSES)],
    yticklabels=[IDX_TO_TARGET[i] for i in range(NUM_CLASSES)]
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Multiclass Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix_multiclass.png', dpi=150)
plt.show()

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>9. Save Models</b></p>

In [ ]:
# Save everything
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'event_vocab': EVENT_VOCAB,
        'embed_dim': 64,
        'hidden_dim': 64,
        'num_layers': 3,
        'output_dim': 64,
        'num_classes': NUM_CLASSES
    }
}, 'gnn_multiclass.pt')

with open('multiclass_config.pkl', 'wb') as f:
    pickle.dump({
        'event_encoder': event_encoder,
        'target_classes': TARGET_CLASSES,
        'target_to_idx': TARGET_TO_IDX,
        'idx_to_target': IDX_TO_TARGET,
        'exclude_apply': EXCLUDE_APPLY_FROM_INPUT
    }, f)

xgb_model.save_model('xgboost_multiclass.json')

print("✅ Saved:")
print("   - gnn_multiclass.pt")
print("   - multiclass_config.pkl")
print("   - xgboost_multiclass.json")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>10. Inference Pipeline</b></p>

In [ ]:
class MulticlassPipeline:
    """
    Predict which specific product a user will apply for.
    """
    
    def __init__(self, gnn_path, config_path, xgb_path, device='cpu'):
        self.device = torch.device(device)
        
        # Load config
        with open(config_path, 'rb') as f:
            config = pickle.load(f)
        
        self.event_encoder = config['event_encoder']
        self.target_classes = config['target_classes']
        self.idx_to_target = config['idx_to_target']
        self.exclude_apply = config['exclude_apply']
        
        # Load GNN
        ckpt = torch.load(gnn_path, map_location=device)
        cfg = ckpt['config']
        
        encoder = GNNEncoder(
            event_vocab=cfg['event_vocab'],
            embed_dim=cfg['embed_dim'],
            hidden_dim=cfg['hidden_dim'],
            num_layers=cfg['num_layers'],
            output_dim=cfg['output_dim']
        )
        self.model = GNNClassifier(encoder, num_classes=cfg['num_classes'])
        self.model.load_state_dict(ckpt['model_state_dict'])
        self.model.to(self.device).eval()
        
        # Load XGBoost
        self.xgb = xgb.XGBClassifier()
        self.xgb.load_model(xgb_path)
        
        # Graph builder
        self.graph_builder = SimpleGraphBuilder(self.event_encoder, exclude_apply=self.exclude_apply)
        
        print(f"✅ Pipeline loaded ({len(self.target_classes)} classes)")
    
    @torch.no_grad()
    def predict(self, events):
        """
        Predict which product user will apply for.
        
        Args:
            events: list of event names (the user journey)
        
        Returns:
            dict with prediction and probabilities for each class
        """
        # Filter out Apply events (they shouldn't be in input)
        if self.exclude_apply:
            events = [e for e in events if not e.startswith('Apply')]
        
        # Check unseen
        known = set(self.event_encoder.classes_)
        unseen = set(events) - known
        if unseen:
            print(f"⚠️ Unseen events → <UNK>: {unseen}")
        
        # Build graph
        session_df = pd.DataFrame({'Event': events})
        graph = self.graph_builder.build_graph(session_df)
        
        # Get embedding
        batch = next(iter(DataLoader([graph], batch_size=1))).to(self.device)
        _, emb = self.model(batch)
        
        # Predict
        probs = self.xgb.predict_proba(emb.cpu().numpy())[0]
        pred_idx = probs.argmax()
        
        return {
            'prediction': self.idx_to_target[pred_idx],
            'confidence': float(probs[pred_idx]),
            'probabilities': {self.idx_to_target[i]: float(p) for i, p in enumerate(probs)}
        }


# Initialize
pipeline = MulticlassPipeline(
    'gnn_multiclass.pt',
    'multiclass_config.pkl',
    'xgboost_multiclass.json',
    device='mps' if torch.backends.mps.is_available() else 'cpu'
)

In [ ]:
# Test predictions
print("="*60)
print("TESTING PREDICTIONS")
print("="*60)

# Credit card research
r = pipeline.predict(['ViewCreditCardRates', 'CreditCardBenefits', 'CompareCreditCards'])
print(f"\nCredit card journey:")
print(f"  Prediction: {r['prediction']}")
print(f"  Confidence: {r['confidence']:.4f}")
print(f"  Top 3 probabilities:")
top3 = sorted(r['probabilities'].items(), key=lambda x: x[1], reverse=True)[:3]
for name, prob in top3:
    print(f"    {name}: {prob:.4f}")

In [ ]:
# Savings journey
r = pipeline.predict(['ViewSavingsOptions', 'CompareSavingsRates', 'SavingsCalculator', 'FAQSavingsInterestRate'])
print(f"\nSavings journey:")
print(f"  Prediction: {r['prediction']}")
print(f"  Confidence: {r['confidence']:.4f}")

In [ ]:
# Auto loan journey
r = pipeline.predict(['ViewAutoLoanOptions', 'AutoLoanCalculator', 'CompareAutoRates', 'AutoLoanInquiry'])
print(f"\nAuto loan journey:")
print(f"  Prediction: {r['prediction']}")
print(f"  Confidence: {r['confidence']:.4f}")

In [ ]:
# Routine banking (no application expected)
r = pipeline.predict(['AccountOverview', 'ViewChecking', 'PayBills', 'ViewStatements'])
print(f"\nRoutine banking:")
print(f"  Prediction: {r['prediction']}")
print(f"  Confidence: {r['confidence']:.4f}")

<hr style="height:1px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>Summary</b></p>

<ol style = 'font-size:16px;font-family:Arial'>In this notebook we had below multiclass targets:
<ol style="font-size:16px;font-family:Arial">  
     <li>No Application (Other)</li>
<li>ApplyCreditCard</li>
<li>ApplySavingsAccount</li>
<li>ApplyAutoLoan</li>
<li>ApplyCheckingAccount</li>
     <li>ApplyPersonalLoan</li>
     <li>ApplyMortgage</li>
     <li>ApplyTeenChecking</li>
     </ol>

<p style = 'font-size:18px;font-family:Arial'><b>Usage</b> <p style = 'font-size:16px;font-family:Arial'>
<code>
result = pipeline.predict(['ViewCreditCardRates', 'CompareCreditCards']) 
print(result['prediction'])  # 'ApplyCreditCard' 
print(result['probabilities'])  # {class: prob, ...}
</code>

<p style = 'font-size:18px;font-family:Arial'><b>Notes</b>
    <ol style = 'font-size:16px;font-family:Arial'>
<li>Apply events are excluded from input features to prevent data leakage                 </li>
<li>Model predicts based on the journey before the application                   </li>
<li>Class weights handle imbalanced data                              </li>


<footer style="padding-bottom:35px; border-bottom:3px solid">
  <div style="float:right; margin-top:14px">Copyright © Teradata - 2026. All Rights Reserved</div>
</footer>